### Maintainable PySpark and Data Tests
Objectives
- Organise existing transformations into reusable functions.
- Detect invalid records and preserve rejection reasons.
- Test behaviour using explicit expected results.
- Build the unfinished franchise pipeline directly in this structure.

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F



#### 3.1 - Turning existing code into functions 

In [0]:
%sql

DESCRIBE TABLE samples.bakehouse.sales_transactions

In [0]:
%sql
-- Check whether there are any NULL values in key 
SELECT *
from samples.bakehouse.sales_transactions
WHERE transactionID IS NULL 

In [0]:
%sql

-- Check whether there are duplicate key
SELECT transactionID
FROM samples.bakehouse.sales_transactions
GROUP BY transactionID
HAVING COUNT(transactionID) > 1 

In [0]:
tables = [
    'sales_transactions',
    'sales_franchises',
    'sales_customers'
]

for t in tables :
    df = spark.read.table(f'samples.bakehouse.{t}')
    print(f'table name: {t}, row count: {df.count()}, column count: {len(df.columns)}')
    df.printSchema()
    print('-'*20)

In [0]:
# Create functions to standarize the data
def standarize_transactions(df:DataFrame) -> DataFrame :
    cols = ['transactionID', 'customerID', 'franchiseID', 'dateTime', 'product', 'quantity', 'unitPrice', 'totalPrice']
    return (
        df.select(cols)
        .withColumn('transactionID', F.col('transactionID').cast('string'))
        .withColumn('customerID', F.col('customerID').cast('string'))
        .withColumn('franchiseID', F.col('franchiseID').cast('string'))
        .withColumn('transaction_date', F.col('dateTime').cast('date'))
        .withColumn('unitPrice', F.col('unitPrice').cast('decimal(10,2)'))
        .withColumn('totalPrice', F.col('totalPrice').cast('decimal(10,2)'))
        .withColumn('calculated_total', F.col('unitPrice')*F.col('quantity').cast('decimal(10,2)'))
    )
        
def strandarize_customers(df:DataFrame) -> DataFrame :
    cols = ['customerID', 'first_name', 'last_name', 'city', 'state', 'country', 'continent', 'gender']
    return (
        df.select(cols)
        .withColumn('customerID', F.col('customerID').cast('string'))
    )

def strandarize_franchises(df:DataFrame) -> DataFrame :
    cols = ['franchiseID', 'name', 'city', 'district', 'country', 'size']
    return (
        df.select(cols)
        .withColumn('franchiseID', F.col('franchiseID').cast('string'))
        .withColumnRenamed('name', 'franchiseName')
    )

In [0]:
df_raw_transactions = spark.read.table('samples.bakehouse.sales_transactions')
df_transactions = standarize_transactions(df_raw_transactions)

df_raw_customers = spark.read.table('samples.bakehouse.sales_customers')
df_customers = strandarize_customers(df_raw_customers)

df_raw_franchises = spark.read.table('samples.bakehouse.sales_franchises')
df_franchises = strandarize_franchises(df_raw_franchises)

In [0]:
df_transactions.show(5)
df_customers.show(5)
df_franchises.show(5)

#### 3.2 - Validate sales and preserve rejects 

In [0]:
%sql 
-- deduplication the identical rows 
SELECT *, COUNT(*) AS duplicate_count
FROM samples.bakehouse.sales_transactions
GROUP BY ALL
HAVING duplicate_count > 1 

In [0]:
%sql
-- different bussiness rows sharing same transaction key 
SELECT transactionID, COUNT(DISTINCT struct(customerID, franchiseID, dateTime, product, quantity, unitPrice, totalPrice)) AS different_versions
FROM samples.bakehouse.sales_transactions
GROUP BY transactionID
HAVING different_versions > 1 

In [0]:
def check_global_quality(df:DataFrame, key:str) -> tuple[DataFrame, dict[str, int]] :
    global_quality = {
        'null_key': (df.filter(F.col(key).isNull()).count()),
        'identical_rows':(
            df.groupBy(*df.columns).count().filter(F.col('count') > 1).count()
        )
    }

    if global_quality['identical_rows'] > 0:
        print('There are duplicate rows in the dataset')

        dup_rows = df.groupBy(*df.columns).count().filter(F.col('count') > 1)

        df_dedup = df.dropDuplicates()
        dedup_rows = df_dedup.count()

        input_rows = df.count()
        
        removed_dup_rows = dup_rows.select(F.sum(F.col('count') -1).alias('removed')).first()['removed']

        assert input_rows == dedup_rows + removed_dup_rows
    else:
        print('No identical duplicate rows in the dataset')
        df_dedup = df

    # Check if there are different rows sharing the same key after deduplication the identical rows
    dup_keys_rows = df_dedup.groupBy('transactionID').count().filter(F.col('count') > 1)
    if dup_keys_rows.count() > 0:
        raise ValueError('There are different rows sharing the same transactionID')

    return df_dedup, global_quality

If ValeuError raised, the data validation should be stopped and check the rows.

In [0]:
df_dedup = check_global_quality(df_transactions, 'transactionID')

Quality rules for rows :
- missing_key : Required keys present; text keys not blank
- invalid_date : Date convertible and non-null
- invalide_product : product name non string
- invalid_quantity : Quantity convertible, non-null, and > 0
- invalid_price : Prices convertible, non-null, and ≥ 0
- price_mismatch : Absolute difference: total vs quantity * unit price <> tolerance
- unknown_customer, unknown_franchise : Customer and franchise exist in transactions tables but not in customer and franchise tables

In [0]:
def split_valid_rejected(df_dedup, df_customers, df_franchises, tolerance=0.01):
    return(
        df_transactions
        .join(
            df_customers.select('customerID').distinct().withColumn('customer_exists', F.lit(True)),
            on='customerID',
            how='left'
        )
        .join(
            df_franchises.select('franchiseID').distinct().withColumn('franchise_exists', F.lit(True)),
            on='franchiseID',
            how='left'
        )
        .withColumn(
            'dq_reason',

            # Required keys
            F.when(
                F.col('transactionID').isNull()| (F.trim(F.col('transactionID')) == '')
                | F.col('customerID').isNull() | (F.trim(F.col('customerID')) == '')
                | F.col('franchiseID').isNull() | (F.trim(F.col('franchiseID')) == ''),
                F.lit('missing_key')
            )
            # Date must be convertible and non-null
            .when(F.col('dateTime').isNull()
                | F.col('transaction_date').isNull(),
                F.lit('invalid_date')
            )
            # Product must exist and not be blank
            .when(F.col('product').isNull()
                | (F.trim(F.col('product')) == ''),
                F.lit('invalid_product')
            )
            # Quantity > 0
            .when(F.col('quantity').isNull()
                | (F.col('quantity') <= 0),
                F.lit('invalid_quantity')
            )
            # Prices >= 0
            .when(F.col('unitPrice').isNull()
                | (F.col('unitPrice') < 0)
                | F.col('totalPrice').isNull()
                | (F.col('totalPrice') < 0),
                F.lit('invalid_price')
            )
            # Difference greater than tolerance
            .when(F.abs(
                    F.col('totalPrice')
                    - F.col('calculated_total')
                ) > tolerance,
                F.lit('price_mismatch')
            )
            # Unknown customers and unknown franchises
            .when(
                F.col('customer_exists').isNull(),
                F.lit('unknown_customer')
            )
            .when(
                F.col('franchise_exists').isNull(),
                F.lit('unknown_franchise')
            )        
            .otherwise(F.lit(None))
        )
        .withColumn(
            'dq_status',
            F.when(
                F.col('dq_reason').isNull(),
                F.lit('VALID')
            ).otherwise(F.lit('INVALID'))
        )
        .withColumn(
            'checked_at',
            F.current_timestamp()
        )
    )


In [0]:
df_transactions_dedup = df_transactions.dropDuplicates()
check_df = split_valid_rejected(df_transactions_dedup, df_customers, df_franchises)
check_df.show(20)

In [0]:
valid_df = check_df.filter(F.col('dq_status') == 'VALID')
invalid_df = check_df.filter(F.col('dq_status') == 'INVALID')
valid_df.show(5)
invalid_df.show(5)

In [0]:
# Verify row counts after deduplication 
input_rows = df_transactions.count()
output_rows = valid_df.count() + invalid_df.count()
assert input_rows == output_rows, 'Input and output rows do not match'

#### 3.3 - Test with small, controlled datasets 

In [0]:
from datetime import datetime
from decimal import Decimal

In [0]:
# Create a manual test DataFrame for standardisation and validation tests
test_data = [
    # 3 valid rows
    (1003103, 2000075, 3000047, datetime(2024, 5, 4, 23, 44),
     'Pearly Pies', 28, Decimal('3.00'), Decimal('84.00')),

    (1000864, 2000290, 3000047, datetime(2024, 5, 6, 20, 46),
     'Golden Gate Ginger', 4, Decimal('3.00'), Decimal('12.00')),

    (1003222, 2000179, 3000046, datetime(2024, 5, 8, 10, 19),
     'Outback Oatmeal', 36, Decimal('3.00'), Decimal('108.00')),

    # Strict duplicate of the 1st row
    (1003103, 2000075, 3000047, datetime(2024, 5, 4, 23, 44),
     'Pearly Pies', 28, Decimal('3.00'), Decimal('84.00')),

    # dateTime NULL
    (1003230, 2000248, 3000046, None,
     'Tokyo Tidbits', 54, Decimal('3.00'), Decimal('162.00')),

    # product empty
    (1004001, 2000179, 3000046, datetime(2024, 5, 8, 10, 19),
     '', 10, Decimal('3.00'), Decimal('30.00')),

    # quantity = 0
    (1004002, 2000179, 3000046, datetime(2024, 5, 8, 11, 20),
     'Pearly Pies', 0, Decimal('3.00'), Decimal('0.00')),

    # Price mismatch
    (1004005, 2000179, 3000046, datetime(2024, 5, 8, 14, 20),
     'Tokyo Tidbits', 2, Decimal('3.00'), Decimal('10.00')),

    # Negative unitPrice
    (1004004, 2000179, 3000046, datetime(2024, 5, 8, 13, 20),
     'Outback Oatmeal', 2, Decimal('-3.00'), Decimal('-6.00')),

    # Missing transactionID
    (None, 2000179, 3000046, datetime(2024, 5, 8, 15, 20),
     'Pearly Pies', 2, Decimal('3.00'), Decimal('6.00')),

    # Unknown customer
    (1004006, 9999999, 3000046, datetime(2024, 5, 8, 16, 20),
     'Pearly Pies', 2, Decimal('3.00'), Decimal('6.00')),

    # Unknown franchise
    (1004007, 2000179, 9999999, datetime(2024, 5, 8, 17, 20),
     'Pearly Pies', 2, Decimal('3.00'), Decimal('6.00')),
]

columns = [
    'transactionID',
    'customerID',
    'franchiseID',
    'dateTime',
    'product',
    'quantity',
    'unitPrice',
    'totalPrice'
]

df_test = spark.createDataFrame(test_data, columns)

df_test.show(truncate=False)

In [0]:
test_df_std = standarize_transactions(df_test)
test_df_std.show()
test_df_std.printSchema()

In [0]:
test_df_dedup, global_quality_dict = check_global_quality(test_df_std, 'transactionID')

assert global_quality_dict['identical_rows'] == 1
assert global_quality_dict['null_key'] == 1
assert test_df_dedup.count() == 11

print(global_quality_dict)

In [0]:
test_check_df = split_valid_rejected(test_df_dedup, df_customers, df_franchises)
test_check_df.show()

In [0]:
assert test_check_df.filter(F.col('dq_reason') == 'invalid_date').count() == 1
assert test_check_df.filter(F.col('dq_reason') == 'invalid_product').count() == 1
assert test_check_df.filter(F.col('dq_reason') == 'invalid_quantity').count() == 1
assert test_check_df.filter(F.col('dq_reason') == 'invalid_price').count() == 1
assert test_check_df.filter(F.col('dq_reason') == 'price_mismatch').count() == 1
assert test_check_df.filter(F.col('dq_reason') == 'missing_key').count() == 1
assert test_check_df.filter(F.col('dq_reason') == 'unknown_customer').count() == 1
assert test_check_df.filter(F.col('dq_reason') == 'unknown_franchise').count() == 1

In [0]:
test_valid_df = test_check_df.filter(F.col('dq_status') == 'VALID')
test_reject_df = test_check_df.filter(F.col('dq_status') == 'INVALID')
test_valid_df.show()
test_reject_df.show()

dedup_rows = test_df_dedup.count()
output_rows = test_valid_df.count() + test_reject_df.count()

In [0]:
assert test_valid_df.count() == 3
assert test_reject_df.count() == 8
assert dedup_rows == output_rows, 'Input and output rows do not match'